# Sudoku RL SFT Training

This notebook builds a synthetic step-level Sudoku dataset for multiple difficulty levels, fine-tunes a chat model with supervised fine-tuning (SFT), and evaluates the trained model through real OpenEnv episodes.

Artifacts written by the notebook:
- `artifacts/sft_data/train_puzzles.jsonl`
- `artifacts/sft_data/train_sft_steps.jsonl`
- `artifacts/sft_data/test_cases.jsonl`
- `artifacts/sft_data/test_episode_results.jsonl`
- `artifacts/sft_data/test_episode_steps.jsonl`

In [ ]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

try:
    import trl  # noqa: F401
    import datasets  # noqa: F401
    import peft  # noqa: F401
    import transformers  # noqa: F401
    import openenv  # noqa: F401
    print("trl, datasets, peft, transformers, and openenv are available in this kernel.")
except ModuleNotFoundError:
    print("One or more notebook dependencies are missing.")
    print("Run the next install cell with %pip, then restart the kernel.")


In [ ]:
# %pip install -q torch transformers accelerate datasets trl peft openenv-core[core] pandas


In [ ]:
import asyncio
import inspect
import json
import os
import random
import re
import subprocess
import textwrap
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from datasets import Dataset
from openenv.core.containers.runtime.providers import LocalDockerProvider
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

from sudoku_rl import SudokuRlAction, SudokuRlEnv
from sudoku_rl.sudoku_logic import (
    build_initial_puzzle,
    calculate_score_step,
    copy_grid,
    format_board_text,
    generate_valid_sudoku,
)

MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
IMAGE_NAME = os.getenv("IMAGE_NAME", "sudoku_rl:latest")
BASE_URL = os.getenv("SUDOKU_BASE_URL", "").strip()
SEED = int(os.getenv("SEED", "7"))
DIFFICULTY_LEVELS = [
    int(item)
    for item in os.getenv("DIFFICULTY_LEVELS", "5,10,20,25,35").split(",")
    if item.strip()
]
MIN_EXAMPLES_PER_DIFFICULTY = int(os.getenv("MIN_EXAMPLES_PER_DIFFICULTY", "8"))
EVAL_CASES_PER_DIFFICULTY = int(os.getenv("EVAL_CASES_PER_DIFFICULTY", "5"))
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "64"))
MAX_STEPS_MULTIPLIER = float(os.getenv("MAX_STEPS_MULTIPLIER", "3.0"))
VERBOSE_EVAL = os.getenv("VERBOSE_EVAL", "1").strip().lower() not in {"0", "false", "no"}
CONNECT_TIMEOUT_S = float(os.getenv("CONNECT_TIMEOUT_S", "60"))
MESSAGE_TIMEOUT_S = float(os.getenv("MESSAGE_TIMEOUT_S", "1800"))
DOCKER_WAIT_TIMEOUT_S = float(os.getenv("DOCKER_WAIT_TIMEOUT_S", "180"))
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "artifacts/sft_checkpoints"))
ARTIFACT_DIR = Path(os.getenv("ARTIFACT_DIR", "artifacts/sft_data"))

SYSTEM_PROMPT = textwrap.dedent(
    """
    You are solving a Sudoku puzzle one move at a time.

    Return exactly one JSON object with this schema:
    {"row": <1-9>, "column": <1-9>, "value": <1-9>}

    Rules:
    - Choose exactly one editable empty cell.
    - Use only digits from 1 to 9.
    - Do not include markdown, explanations, or extra text.
    """
).strip()

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print({
    "model_name": MODEL_NAME,
    "difficulty_levels": DIFFICULTY_LEVELS,
    "min_examples_per_difficulty": MIN_EXAMPLES_PER_DIFFICULTY,
    "eval_cases_per_difficulty": EVAL_CASES_PER_DIFFICULTY,
    "artifact_dir": str(ARTIFACT_DIR.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
    "verbose_eval": VERBOSE_EVAL,
})


In [ ]:
def json_compact(payload: Any) -> str:
    return json.dumps(payload, ensure_ascii=True, separators=(",", ":"))


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=True) + "\n")


def candidate_values(board: list[list[int]], row_index: int, column_index: int) -> list[int]:
    if board[row_index][column_index] != 0:
        return []

    used_values = set(board[row_index])
    used_values.update(board[current_row][column_index] for current_row in range(9))

    box_row_start = (row_index // 3) * 3
    box_column_start = (column_index // 3) * 3
    for box_row in range(box_row_start, box_row_start + 3):
        for box_column in range(box_column_start, box_column_start + 3):
            used_values.add(board[box_row][box_column])

    return [value for value in range(1, 10) if value not in used_values]


def choose_next_solution_step(board: list[list[int]], solution: list[list[int]]) -> tuple[int, int, int, list[int]]:
    ranked_choices: list[tuple[int, int, int, list[int]]] = []
    for row_index in range(9):
        for column_index in range(9):
            if board[row_index][column_index] != 0:
                continue
            candidates = candidate_values(board, row_index, column_index)
            ranked_choices.append((len(candidates), row_index, column_index, candidates))

    ranked_choices.sort(key=lambda item: (item[0], item[1], item[2]))
    _, row_index, column_index, candidates = ranked_choices[0]
    return row_index, column_index, solution[row_index][column_index], candidates


def find_row_conflict_value(
    board: list[list[int]],
    row_index: int,
    column_index: int,
    correct_value: int,
) -> int | None:
    del column_index
    for value in board[row_index]:
        if value != 0 and value != correct_value:
            return value
    return None


def build_user_prompt(
    board: list[list[int]],
    difficulty: int,
    history: list[str] | None = None,
    *,
    last_env_message: str = "Puzzle ready.",
    invalid_cells: list[list[int]] | None = None,
    score: int = 0,
    mistakes: int = 0,
    moves: int = 0,
) -> str:
    history_block = "\n".join((history or [])[-6:]) if history else "None"
    invalid_block = invalid_cells if invalid_cells else []
    return textwrap.dedent(
        f"""
        Difficulty: {difficulty} empty cells
        Current score: {score}
        Moves attempted: {moves}
        Mistakes: {mistakes}
        Last environment message: {last_env_message}
        Invalid cells (0-based [row, col]): {invalid_block}

        Current board:
        {format_board_text(board)}

        Recent move history:
        {history_block}

        Return the next move as JSON only.
        """
    ).strip()


def build_messages(user_prompt: str, completion: str | None = None) -> list[dict[str, str]]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    if completion is not None:
        messages.append({"role": "assistant", "content": completion})
    return messages


def build_history_entry(
    *,
    step_index: int,
    action: dict[str, int],
    reward: int,
    status: str,
    message: str,
) -> str:
    return (
        f"step={step_index} action={json_compact(action)} reward={reward} "
        f"status={status} message={message}"
    )


def build_training_example(
    *,
    board: list[list[int]],
    difficulty: int,
    history: list[str],
    last_env_message: str,
    invalid_cells: list[list[int]] | None,
    score: int,
    mistakes: int,
    moves: int,
    puzzle_id: str,
    puzzle_seed: int,
    step_index: int,
    solution: list[list[int]],
    target_action: dict[str, int],
    candidate_values_for_target: list[int],
    example_type: str,
    branch_from_step: int | None = None,
    invalid_action: dict[str, int] | None = None,
    invalid_reason: str = "",
) -> dict[str, Any]:
    invalid_cells = invalid_cells or []
    completion = json_compact(target_action)
    user_prompt = build_user_prompt(
        board,
        difficulty,
        history,
        last_env_message=last_env_message,
        invalid_cells=invalid_cells,
        score=score,
        mistakes=mistakes,
        moves=moves,
    )
    return {
        "split": "train",
        "difficulty": difficulty,
        "puzzle_id": puzzle_id,
        "puzzle_seed": puzzle_seed,
        "step_index": step_index,
        "branch_from_step": branch_from_step,
        "example_type": example_type,
        "prompt": user_prompt,
        "completion": completion,
        "messages": build_messages(user_prompt, completion),
        "board": copy_grid(board),
        "solution": solution,
        "target_row": target_action["row"],
        "target_column": target_action["column"],
        "target_value": target_action["value"],
        "candidate_values": candidate_values_for_target,
        "candidate_count": len(candidate_values_for_target),
        "last_env_message": last_env_message,
        "invalid_cells": invalid_cells,
        "score": score,
        "moves": moves,
        "mistakes": mistakes,
        "history": history[-6:],
        "invalid_action": invalid_action,
        "invalid_reason": invalid_reason,
    }


def build_synthetic_puzzle(example_seed: int, difficulty: int) -> tuple[list[list[int]], list[list[int]]]:
    rng = random.Random(example_seed)
    solution = generate_valid_sudoku(rng)
    puzzle = build_initial_puzzle(solution, holes=difficulty, rng=rng)
    return puzzle, solution


def simulate_solution_path(
    puzzle: list[list[int]],
    solution: list[list[int]],
    *,
    difficulty: int,
    puzzle_id: str,
    puzzle_seed: int,
) -> list[dict[str, Any]]:
    board = copy_grid(puzzle)
    history: list[str] = []
    rows: list[dict[str, Any]] = []
    valid_step_index = 0
    score_step = calculate_score_step(puzzle)
    score = 0
    moves = 0
    mistakes = 0
    last_env_message = "Puzzle ready. Click any box to choose the row and column you want to fill."

    while any(value == 0 for row in board for value in row):
        valid_step_index += 1
        row_index, column_index, value, candidates = choose_next_solution_step(board, solution)
        valid_action = {
            "row": row_index + 1,
            "column": column_index + 1,
            "value": value,
        }

        rows.append(
            build_training_example(
                board=board,
                difficulty=difficulty,
                history=history,
                last_env_message=last_env_message,
                invalid_cells=[],
                score=score,
                mistakes=mistakes,
                moves=moves,
                puzzle_id=puzzle_id,
                puzzle_seed=puzzle_seed,
                step_index=valid_step_index,
                solution=solution,
                target_action=valid_action,
                candidate_values_for_target=candidates,
                example_type="valid_move",
            )
        )

        row_conflict_value = find_row_conflict_value(board, row_index, column_index, value)
        if row_conflict_value is not None:
            row_conflict_board = copy_grid(board)
            row_conflict_board[row_index][column_index] = row_conflict_value
            row_conflict_action = {
                "row": row_index + 1,
                "column": column_index + 1,
                "value": row_conflict_value,
            }
            row_conflict_reason = "Error: number in row."
            row_conflict_message = (
                f"{row_conflict_reason} Score change: {-score_step}. Total score: {score - score_step}."
            )
            row_conflict_history = history + [
                build_history_entry(
                    step_index=valid_step_index,
                    action=row_conflict_action,
                    reward=-score_step,
                    status="invalid_move",
                    message=row_conflict_message,
                )
            ]
            rows.append(
                build_training_example(
                    board=row_conflict_board,
                    difficulty=difficulty,
                    history=row_conflict_history,
                    last_env_message=row_conflict_message,
                    invalid_cells=[[row_index, column_index]],
                    score=score - score_step,
                    mistakes=mistakes + 1,
                    moves=moves + 1,
                    puzzle_id=puzzle_id,
                    puzzle_seed=puzzle_seed,
                    step_index=valid_step_index,
                    branch_from_step=valid_step_index,
                    solution=solution,
                    target_action=valid_action,
                    candidate_values_for_target=candidates,
                    example_type="row_conflict_recovery",
                    invalid_action=row_conflict_action,
                    invalid_reason=row_conflict_reason,
                )
            )

        board[row_index][column_index] = value
        score += score_step
        moves += 1
        solved = not any(cell == 0 for row in board for cell in row)
        if solved:
            last_env_message = f"Sudoku solved completely. Final score: {score}."
        else:
            last_env_message = (
                f"Valid move: this number can be placed here. Updated row {row_index + 1}, "
                f"column {column_index + 1} with {value}. Score change: +{score_step}. Total score: {score}."
            )

        history.append(
            build_history_entry(
                step_index=valid_step_index,
                action=valid_action,
                reward=score_step,
                status="solved" if solved else "in_progress",
                message=last_env_message,
            )
        )
        history = history[-6:]

        if solved:
            continue

        next_row_index, next_column_index, next_value, next_candidates = choose_next_solution_step(board, solution)
        next_valid_action = {
            "row": next_row_index + 1,
            "column": next_column_index + 1,
            "value": next_value,
        }
        same_cell_reason = "Other error: the selected cell was already updated and cannot be changed."
        same_cell_message = (
            f"{same_cell_reason} Score change: {-score_step}. Total score: {score - score_step}."
        )
        same_cell_history = history + [
            build_history_entry(
                step_index=valid_step_index,
                action=valid_action,
                reward=-score_step,
                status="invalid_move",
                message=same_cell_message,
            )
        ]
        rows.append(
            build_training_example(
                board=board,
                difficulty=difficulty,
                history=same_cell_history,
                last_env_message=same_cell_message,
                invalid_cells=[],
                score=score - score_step,
                mistakes=mistakes + 1,
                moves=moves + 1,
                puzzle_id=puzzle_id,
                puzzle_seed=puzzle_seed,
                step_index=valid_step_index + 1,
                branch_from_step=valid_step_index,
                solution=solution,
                target_action=next_valid_action,
                candidate_values_for_target=next_candidates,
                example_type="same_cell_recovery",
                invalid_action=valid_action,
                invalid_reason=same_cell_reason,
            )
        )

    return rows


def generate_synthetic_sft_data(
    difficulty_levels: list[int],
    min_examples_per_difficulty: int,
    *,
    seed: int = SEED,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    train_steps: list[dict[str, Any]] = []
    train_puzzles: list[dict[str, Any]] = []
    next_seed = seed

    for difficulty in difficulty_levels:
        for example_index in range(min_examples_per_difficulty):
            puzzle_seed = next_seed
            next_seed += 1
            puzzle_id = f"train-d{difficulty}-p{example_index:03d}"
            puzzle, solution = build_synthetic_puzzle(puzzle_seed, difficulty)
            train_puzzles.append({
                "split": "train",
                "difficulty": difficulty,
                "puzzle_id": puzzle_id,
                "puzzle_seed": puzzle_seed,
                "puzzle": puzzle,
                "solution": solution,
                "empty_cells": difficulty,
            })
            train_steps.extend(
                simulate_solution_path(
                    puzzle,
                    solution,
                    difficulty=difficulty,
                    puzzle_id=puzzle_id,
                    puzzle_seed=puzzle_seed,
                )
            )

    return train_steps, train_puzzles


def generate_eval_case_specs(
    difficulty_levels: list[int],
    *,
    cases_per_difficulty: int = 5,
    seed: int = SEED + 10_000,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    next_seed = seed

    for difficulty in difficulty_levels:
        for case_index in range(cases_per_difficulty):
            case_seed = next_seed
            next_seed += 1
            preview_puzzle, preview_solution = build_synthetic_puzzle(case_seed, difficulty)
            rows.append({
                "split": "test",
                "difficulty": difficulty,
                "case_id": f"test-d{difficulty}-c{case_index:02d}",
                "case_seed": case_seed,
                "expected_initial_puzzle": preview_puzzle,
                "expected_solution": preview_solution,
                "empty_cells": difficulty,
            })

    return rows


In [ ]:
train_steps, train_puzzles = generate_synthetic_sft_data(
    DIFFICULTY_LEVELS,
    MIN_EXAMPLES_PER_DIFFICULTY,
    seed=SEED,
)
test_cases = generate_eval_case_specs(
    DIFFICULTY_LEVELS,
    cases_per_difficulty=EVAL_CASES_PER_DIFFICULTY,
    seed=SEED + 10_000,
)

write_jsonl(ARTIFACT_DIR / "train_puzzles.jsonl", train_puzzles)
write_jsonl(ARTIFACT_DIR / "train_sft_steps.jsonl", train_steps)
write_jsonl(ARTIFACT_DIR / "test_cases.jsonl", test_cases)

train_puzzle_summary = pd.DataFrame(train_puzzles).groupby("difficulty").size().rename("train_puzzles")
train_step_summary = pd.DataFrame(train_steps).groupby("difficulty").size().rename("train_sft_steps")
test_case_summary = pd.DataFrame(test_cases).groupby("difficulty").size().rename("test_cases")
example_type_summary = pd.DataFrame(train_steps).groupby(["difficulty", "example_type"]).size().unstack(fill_value=0)
summary = pd.concat([train_puzzle_summary, train_step_summary, test_case_summary], axis=1).join(example_type_summary, how="left").fillna(0).astype(int)
summary


## SFT setup

The synthetic generator above is parametric on `difficulty_levels` and `min_examples_per_difficulty`, so you can reuse the same functions with a different difficulty array or a larger dataset budget without rewriting the notebook.

In [ ]:
def render_training_text(messages: list[dict[str, str]], tokenizer: AutoTokenizer) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    formatted_messages = []
    for message in messages:
        formatted_messages.append(f"{message['role'].upper()}: {message['content']}")
    return "\n\n".join(formatted_messages)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset_rows = [
    {
        **row,
        "text": render_training_text(row["messages"], tokenizer),
    }
    for row in train_steps
]
train_dataset = Dataset.from_list(train_dataset_rows)

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16
model_kwargs: dict[str, Any] = {"trust_remote_code": True}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"
    model_kwargs["torch_dtype"] = torch.bfloat16 if bf16 else torch.float16
else:
    model_kwargs["torch_dtype"] = torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

sft_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "overwrite_output_dir": True,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "logging_steps": 5,
    "save_strategy": "epoch",
    "report_to": "none",
    "bf16": bf16,
    "fp16": fp16,
    "max_seq_length": 1024,
    "dataset_text_field": "text",
}
supported_sft_keys = set(inspect.signature(SFTConfig.__init__).parameters)
filtered_sft_kwargs = {key: value for key, value in sft_kwargs.items() if key in supported_sft_keys}
skipped_sft_keys = sorted(set(sft_kwargs) - set(filtered_sft_kwargs))
if skipped_sft_keys:
    print("Skipping unsupported SFTConfig args:", skipped_sft_keys)

sft_config = SFTConfig(**filtered_sft_kwargs)
trainer_init_keys = set(inspect.signature(SFTTrainer.__init__).parameters)
trainer_kwargs: dict[str, Any] = {
    "model": model,
    "args": sft_config,
    "train_dataset": train_dataset,
    "peft_config": peft_config,
}
if "tokenizer" in trainer_init_keys:
    trainer_kwargs["tokenizer"] = tokenizer
if "processing_class" in trainer_init_keys:
    trainer_kwargs["processing_class"] = tokenizer
if "dataset_text_field" in trainer_init_keys:
    trainer_kwargs["dataset_text_field"] = "text"

trainer = SFTTrainer(**trainer_kwargs)
print("Training examples:", len(train_dataset))


In [ ]:
train_result = trainer.train()
trainer.model.eval()
if hasattr(trainer.model, "gradient_checkpointing_disable"):
    trainer.model.gradient_checkpointing_disable()
if hasattr(trainer.model.config, "use_cache"):
    trainer.model.config.use_cache = True
if hasattr(trainer.model, "generation_config"):
    trainer.model.generation_config.do_sample = False
    for field_name in ("temperature", "top_p", "top_k"):
        if hasattr(trainer.model.generation_config, field_name):
            setattr(trainer.model.generation_config, field_name, None)

adapter_dir = OUTPUT_DIR / "adapter"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

metrics_path = ARTIFACT_DIR / "train_metrics.json"
metrics_path.write_text(json.dumps(train_result.metrics, indent=2), encoding="utf-8")
train_result.metrics


In [ ]:
ACTION_RE = re.compile(r"\{[^{}]+\}", flags=re.DOTALL)


def parse_action_text(text: str) -> dict[str, int] | None:
    cleaned = (text or "").strip()
    if not cleaned:
        return None

    candidates: list[dict[str, Any]] = []
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            candidates.append(parsed)
    except json.JSONDecodeError:
        pass

    for match in ACTION_RE.finditer(cleaned):
        try:
            parsed = json.loads(match.group(0))
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            candidates.append(parsed)

    row_match = re.search(r'"?row"?\s*[:=]\s*([1-9])', cleaned, flags=re.IGNORECASE)
    column_match = re.search(r'"?(?:column|col)"?\s*[:=]\s*([1-9])', cleaned, flags=re.IGNORECASE)
    value_match = re.search(r'"?(?:value|number)"?\s*[:=]\s*([1-9])', cleaned, flags=re.IGNORECASE)
    if row_match and column_match and value_match:
        candidates.append({
            "row": int(row_match.group(1)),
            "column": int(column_match.group(1)),
            "value": int(value_match.group(1)),
        })

    for candidate in candidates:
        try:
            row = int(candidate["row"])
            column = int(candidate["column"])
            value = int(candidate.get("value", candidate.get("number")))
        except (KeyError, TypeError, ValueError):
            continue
        if 1 <= row <= 9 and 1 <= column <= 9 and 1 <= value <= 9:
            return {"row": row, "column": column, "value": value}

    return None


def render_generation_prompt(messages: list[dict[str, str]], tokenizer: AutoTokenizer) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    formatted_messages = []
    for message in messages:
        formatted_messages.append(f"{message['role'].upper()}: {message['content']}")
    return "\n\n".join(formatted_messages) + "\n\nASSISTANT:"


def prepare_model_for_inference(model: Any) -> None:
    model.eval()
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = True
    if hasattr(model, "generation_config"):
        model.generation_config.do_sample = False
        for field_name in ("temperature", "top_p", "top_k"):
            if hasattr(model.generation_config, field_name):
                setattr(model.generation_config, field_name, None)


def generate_model_output(
    model: Any,
    tokenizer: AutoTokenizer,
    user_prompt: str,
    *,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> str:
    prepare_model_for_inference(model)
    prompt_text = render_generation_prompt(build_messages(user_prompt), tokenizer)
    device = next(model.parameters()).device
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


async def close_env_quietly(env: SudokuRlEnv | None) -> None:
    if env is None:
        return
    try:
        await env.close()
    except subprocess.TimeoutExpired as exc:
        print(f"Environment close timed out after {exc.timeout}s; continuing without failing evaluation.")
    except Exception as exc:
        print(f"Environment close raised {type(exc).__name__}: {exc}")


def print_episode_header(case_id: str, difficulty: int, initial_puzzle_text: str, *, max_steps: int) -> None:
    print(f"\n[EPISODE_START] case_id={case_id} difficulty={difficulty} max_steps={max_steps}")
    print("[INITIAL_PUZZLE]")
    print(initial_puzzle_text)


def print_episode_step(
    *,
    case_id: str,
    step_index: int,
    current_board_text: str,
    model_output: str,
    parsed_action: dict[str, int] | None,
    reward: float,
    points: int,
    status: str,
    message: str,
    next_board_text: str,
    executed: bool,
) -> None:
    print(f"\n[STEP] case_id={case_id} step={step_index} executed={executed} status={status} reward={reward} points={points}")
    print("[CURRENT_STATE]")
    print(current_board_text)
    print("[MODEL_OUTPUT]")
    print(model_output)
    print("[PARSED_ACTION]")
    print(parsed_action)
    print("[ENV_MESSAGE]")
    print(message)
    print("[NEXT_STATE]")
    print(next_board_text)


def print_episode_footer(case_id: str, solved: bool, final_status: str, points: int, steps_taken: int) -> None:
    print(
        f"\n[EPISODE_END] case_id={case_id} solved={solved} final_status={final_status} points={points} steps_taken={steps_taken}"
    )


async def create_env() -> SudokuRlEnv:
    if BASE_URL:
        client = SudokuRlEnv(
            base_url=BASE_URL,
            connect_timeout_s=CONNECT_TIMEOUT_S,
            message_timeout_s=MESSAGE_TIMEOUT_S,
        )
        return await client.connect()

    provider = LocalDockerProvider()
    runtime_base_url = provider.start_container(IMAGE_NAME)
    provider.wait_for_ready(runtime_base_url, timeout_s=DOCKER_WAIT_TIMEOUT_S)
    client = SudokuRlEnv(
        base_url=runtime_base_url,
        connect_timeout_s=CONNECT_TIMEOUT_S,
        message_timeout_s=MESSAGE_TIMEOUT_S,
        provider=provider,
    )
    return await client.connect()


async def run_openenv_episode(
    env: SudokuRlEnv,
    model: Any,
    tokenizer: AutoTokenizer,
    *,
    difficulty: int,
    case_seed: int,
    case_id: str,
    max_steps: int | None = None,
    verbose: bool = VERBOSE_EVAL,
) -> dict[str, Any]:
    history: list[str] = []
    step_logs: list[dict[str, Any]] = []
    result = None
    observation = None

    result = await env.reset(empty_boxes=difficulty, seed=case_seed)
    observation = result.observation
    if max_steps is None:
        max_steps = max(10, int(difficulty * MAX_STEPS_MULTIPLIER))
    if verbose:
        print_episode_header(
            case_id,
            difficulty,
            format_board_text(observation.initial_puzzle),
            max_steps=max_steps,
        )

    for step_index in range(1, max_steps + 1):
        if result.done:
            break

        pre_step_board = copy_grid(observation.board)
        pre_step_board_text = observation.board_text
        initial_puzzle = copy_grid(observation.initial_puzzle)
        initial_puzzle_text = format_board_text(initial_puzzle)

        user_prompt = build_user_prompt(
            pre_step_board,
            difficulty,
            history,
            last_env_message=observation.message,
            invalid_cells=observation.invalid_cells,
            score=observation.score,
            mistakes=observation.mistakes,
            moves=observation.moves,
        )
        model_output = generate_model_output(model, tokenizer, user_prompt)
        parsed_action = parse_action_text(model_output)

        if parsed_action is None:
            step_logs.append({
                "case_id": case_id,
                "difficulty": difficulty,
                "case_seed": case_seed,
                "step_index": step_index,
                "initial_puzzle": initial_puzzle,
                "initial_puzzle_text": initial_puzzle_text,
                "current_board": pre_step_board,
                "current_board_text": pre_step_board_text,
                "prompt": user_prompt,
                "model_output": model_output,
                "parsed_action": None,
                "reward": 0.0,
                "points": observation.score,
                "status": observation.status,
                "message": "Model output could not be parsed into a valid action.",
                "next_board": pre_step_board,
                "next_board_text": pre_step_board_text,
                "done": False,
                "executed": False,
            })
            if verbose:
                print_episode_step(
                    case_id=case_id,
                    step_index=step_index,
                    current_board_text=pre_step_board_text,
                    model_output=model_output,
                    parsed_action=None,
                    reward=0.0,
                    points=observation.score,
                    status=observation.status,
                    message="Model output could not be parsed into a valid action.",
                    next_board_text=pre_step_board_text,
                    executed=False,
                )
            break

        action = SudokuRlAction(
            row=parsed_action["row"],
            column=parsed_action["column"],
            value=parsed_action["value"],
        )
        result = await env.step(action)
        observation = result.observation
        reward = float(result.reward or 0.0)
        history.append(
            f"step={step_index} action={json_compact(parsed_action)} reward={reward} status={observation.status} message={observation.message}"
        )
        history = history[-6:]
        step_logs.append({
            "case_id": case_id,
            "difficulty": difficulty,
            "case_seed": case_seed,
            "step_index": step_index,
            "initial_puzzle": initial_puzzle,
            "initial_puzzle_text": initial_puzzle_text,
            "current_board": pre_step_board,
            "current_board_text": pre_step_board_text,
            "prompt": user_prompt,
            "model_output": model_output,
            "parsed_action": parsed_action,
            "reward": reward,
            "points": observation.score,
            "status": observation.status,
            "message": observation.message,
            "next_board": copy_grid(observation.board),
            "next_board_text": observation.board_text,
            "done": bool(result.done),
            "executed": True,
        })
        if verbose:
            print_episode_step(
                case_id=case_id,
                step_index=step_index,
                current_board_text=pre_step_board_text,
                model_output=model_output,
                parsed_action=parsed_action,
                reward=reward,
                points=observation.score,
                status=observation.status,
                message=observation.message,
                next_board_text=observation.board_text,
                executed=True,
            )

    final_observation = observation
    solved = bool(result.done) if result is not None else False
    if verbose:
        print_episode_footer(
            case_id,
            solved,
            final_observation.status if final_observation is not None else "not_started",
            final_observation.score if final_observation is not None else 0,
            len(step_logs),
        )
    return {
        "case_id": case_id,
        "difficulty": difficulty,
        "case_seed": case_seed,
        "solved": solved,
        "final_status": final_observation.status if final_observation is not None else "not_started",
        "points": final_observation.score if final_observation is not None else 0,
        "steps_taken": len(step_logs),
        "max_steps": max_steps,
        "initial_puzzle": final_observation.initial_puzzle if final_observation is not None else [],
        "final_board": final_observation.board if final_observation is not None else [],
        "episode_steps": step_logs,
    }


async def evaluate_model_with_openenv(
    model: Any,
    tokenizer: AutoTokenizer,
    difficulty_levels: list[int],
    *,
    cases_per_difficulty: int = 5,
    seed: int = SEED + 10_000,
    verbose: bool = VERBOSE_EVAL,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], pd.DataFrame]:
    case_specs = generate_eval_case_specs(
        difficulty_levels,
        cases_per_difficulty=cases_per_difficulty,
        seed=seed,
    )

    episode_results: list[dict[str, Any]] = []
    episode_steps: list[dict[str, Any]] = []
    env = await create_env()

    try:
        for case_spec in case_specs:
            episode = await run_openenv_episode(
                env,
                model,
                tokenizer,
                difficulty=case_spec["difficulty"],
                case_seed=case_spec["case_seed"],
                case_id=case_spec["case_id"],
                verbose=verbose,
            )
            episode_steps.extend(episode.pop("episode_steps"))
            episode_results.append({**case_spec, **episode})
    finally:
        await close_env_quietly(env)

    write_jsonl(ARTIFACT_DIR / "test_episode_results.jsonl", episode_results)
    write_jsonl(ARTIFACT_DIR / "test_episode_steps.jsonl", episode_steps)

    summary = (
        pd.DataFrame(episode_results)
        .groupby("difficulty")
        .agg(
            cases=("case_id", "count"),
            solved=("solved", "sum"),
            average_points=("points", "mean"),
            average_steps=("steps_taken", "mean"),
        )
        .reset_index()
        .sort_values("difficulty")
    )
    return episode_results, episode_steps, summary


In [ ]:
episode_results, episode_steps, evaluation_summary = await evaluate_model_with_openenv(
    trainer.model,
    tokenizer,
    DIFFICULTY_LEVELS,
    cases_per_difficulty=EVAL_CASES_PER_DIFFICULTY,
    seed=SEED + 10_000,
    verbose=VERBOSE_EVAL,
)

evaluation_summary
